# Apéndice · 1 de 4 — Survey del dump de evaluaciones de Lichess

**Esta notebook no entrena nada.** Recorre una parte del dump público de
evaluaciones de Lichess y contesta lo que hay que saber *antes* de comprometer
horas de GPU:

1. ¿Qué trae cada registro, exactamente?
2. ¿Cuántas posiciones hay en el dump entero?
3. ¿A qué profundidad están evaluadas, y cuántas sobreviven a un filtro?
4. ¿Cuánto hay que descargar, y cuánto tarda, para juntar 5, 10 o 20 millones?
5. ¿Qué escalera de brazos entra en la RAM del runtime?

Es el mismo paso que la [notebook 00](../00_survey_dumps.ipynb) hace con los
dumps de PGN, y por el mismo motivo: no hay forma de saber cuántas posiciones
pasan un filtro sin contarlas, y descubrirlo a mitad del etiquetado es caro.

> **Nada de lo que sigue asume el esquema del archivo.** La sección 3 imprime
> el primer registro **crudo** y la 4 detecta dónde está cada cosa. Si el
> formato no es el que se espera, la notebook corta ahí con el registro a la
> vista, en vez de producir un número sin sentido tres horas después.

## Por qué este dump y no otro

El bloque 4 terminó con una conclusión que pide ser puesta a prueba: dos
arquitecturas con priors opuestos llegaron al mismo error y empezaron a
memorizar en el mismo punto, que es la firma de un problema limitado por datos.
Probarlo con el pipeline del proyecto exigiría etiquetar con Stockfish decenas
de millones de posiciones, que es justamente la parte cara. El dump de
evaluaciones de Lichess es la fuente pública más grande de pares *(posición,
evaluación de Stockfish)* ya etiquetados, es **CC0** —la misma licencia que el
requerimiento 5.1 ya cubre para los dumps de partidas— y se baja sin
credenciales.

Además trae **la profundidad de cada evaluación**. Eso permite elegir la calidad de las etiquetas, y es justamente
lo que recorta el volumen.

## 1. Entorno

Runtime de **CPU**: acá no se entrena nada, sólo se descarga y se cuenta.

In [1]:
# Clonar el repositorio e instalar el paquete.
# Idempotente: se puede volver a correr tal cual despues de una desconexion.
import os, sys, subprocess, importlib
from pathlib import Path

REPO_DIR = Path("/content/CEIA-TF-Chess-DL")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "https://github.com/zFonta/CEIA-TF-Chess-DL.git", str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)

os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]"], check=True)

# pip registra el paquete editable con un .pth, y los .pth solo se procesan al
# arrancar el interprete: un kernel que ya esta corriendo no lo ve. Agregar src/
# a sys.path lo hace visible sin tener que reiniciar el runtime.
src = str(REPO_DIR / "src")
if src not in sys.path:
    sys.path.insert(0, src)
importlib.invalidate_caches()

# Stockfish con version fija. Entrenar no lo usa, pero sin el se saltean los 17
# tests de integracion del pipeline, que son la evidencia del requerimiento 3.2.
subprocess.run(["bash", "scripts/setup_stockfish.sh"], check=True)
os.environ["PATH"] = f"{REPO_DIR}/bin:" + os.environ["PATH"]

print("Listo. Directorio de trabajo:", os.getcwd())

Listo. Directorio de trabajo: /content/CEIA-TF-Chess-DL


## 2. El archivo

Se consulta el tamaño con un `HEAD` antes de bajar nada. Ese número es lo que
después permite extrapolar del pedazo leído al dump entero.

In [2]:
import io
import time
from collections import Counter

import requests

URL = 'https://database.lichess.org/lichess_db_eval.jsonl.zst'

# Cuantas lineas recorrer. Con 2 M alcanza para estadisticas estables y son
# unos pocos minutos; subirlo mejora la extrapolacion y cuesta descarga.
LIMITE = 2_000_000

cabeza = requests.head(URL, allow_redirects=True, timeout=60,
                       headers={'User-Agent': 'chessdl/0.1 (CEIA-FIUBA; academico)'})
cabeza.raise_for_status()
COMPRIMIDO_TOTAL = int(cabeza.headers.get('Content-Length', 0))

print(f'{URL}')
print(f'  estado          {cabeza.status_code}')
print(f'  tamaño          {COMPRIMIDO_TOTAL / 1e9:.1f} GB comprimidos'
      if COMPRIMIDO_TOTAL else '  tamaño          no lo informa el servidor')
print(f"  ultima vez      {cabeza.headers.get('Last-Modified', '?')}")

https://database.lichess.org/lichess_db_eval.jsonl.zst
  estado          200
  tamaño          22.1 GB comprimidos
  ultima vez      Thu, 10 Sep 2026 02:44:09 GMT


## 3. El primer registro, crudo

Antes de parsear nada, mirarlo. Esta celda es la que evita escribir un parser
para un formato imaginario.

In [5]:
import zstandard as zstd
import json

class Contador(io.RawIOBase):
    """Envuelve el stream y cuenta bytes COMPRIMIDOS consumidos.

    Es lo que permite extrapolar: sabiendo cuantos bytes comprimidos costaron N
    posiciones, y cuanto pesa el archivo entero, sale el total. `open_pgn_stream`
    del repo hace lo mismo pero esconde el stream crudo, y sin ese contador la
    extrapolacion seria a ojo.
    """

    def __init__(self, fuente):
        self.fuente, self.bytes = fuente, 0

    def read(self, n=-1):
        datos = self.fuente.read(n)
        self.bytes += len(datos)
        return datos

    def readable(self):
        return True


def abrir(url: str):
    """Devuelve (lineas de texto, contador de bytes comprimidos)."""
    respuesta = requests.get(
        url, stream=True, timeout=60,
        headers={'User-Agent': 'chessdl/0.1 (CEIA-FIUBA; academico)'},
    )
    respuesta.raise_for_status()
    contador = Contador(respuesta.raw)
    lector = zstd.ZstdDecompressor().stream_reader(contador, read_size=1 << 20)
    return io.TextIOWrapper(lector, encoding='utf-8', errors='replace'), contador


flujo, _ = abrir(URL)
primera = next(iter(flujo))
flujo.close()

print('--- crudo ---')
print(primera[:600])
print('\n--- parseado ---')
PRIMERO = json.loads(primera)
print(json.dumps(PRIMERO, indent=2, ensure_ascii=False)[:1400])

--- crudo ---
{"fen":"7r/1p3k2/p1bPR3/5p2/2B2P1p/8/PP4P1/3K4 b - -","evals":[{"pvs":[{"cp":69,"line":"f7g7 e6e2 h8d8 e2d2 b7b5 c4b3 g7f6 d1e1 a6a5 a2a3"},{"cp":163,"line":"h8d8 d1e1 a6a5 a2a3 c6d7 e6e7 f7f6 e1f2 b7b5 c4b3"},{"cp":229,"line":"h8a8 d1e1 a6a5 e6h6 f7g7 h6h4 a8d8 c4d3 c6g2 d3f5"},{"cp":231,"line":"h8f8 d1e1 b7b5 c4b3 a6a5 e6h6 f7g7 h6h4 f8e8 e1f2"},{"cp":237,"line":"h8b8 d1e1 a6a5 e6h6 f7g7 h6h4 b8d8 c4d3 c6g2 d3f5"}],"knodes":4189972,"depth":46}]}


--- parseado ---
{
  "fen": "7r/1p3k2/p1bPR3/5p2/2B2P1p/8/PP4P1/3K4 b - -",
  "evals": [
    {
      "pvs": [
        {
          "cp": 69,
          "line": "f7g7 e6e2 h8d8 e2d2 b7b5 c4b3 g7f6 d1e1 a6a5 a2a3"
        },
        {
          "cp": 163,
          "line": "h8d8 d1e1 a6a5 a2a3 c6d7 e6e7 f7f6 e1f2 b7b5 c4b3"
        },
        {
          "cp": 229,
          "line": "h8a8 d1e1 a6a5 e6h6 f7g7 h6h4 a8d8 c4d3 c6g2 d3f5"
        },
        {
          "cp": 231,
          "line": "h8f8 d1e1 b7b5 c4b3 a6a5 e6h6 f7g7 h6

## 4. Dónde está cada cosa

Se ubica el FEN, la lista de evaluaciones y los campos de puntaje **por su
forma**, no por su nombre. Si algo no aparece, la celda corta mostrando el
registro: es un archivo de terceros y el formato puede haber cambiado.

De cada posición se toma **la evaluación más profunda** y, dentro de ella, la
primera línea (`pvs[0]`, la mejor según el motor). Tomar la más profunda es la
decisión de calidad: si una posición está evaluada a 12 y a 36, la de 36 es la
mejor etiqueta disponible y no cuesta nada usarla.

In [6]:
def detectar(registro: dict) -> dict:
    """Ubicar los campos por su forma. Corta si falta alguno."""
    campos = {}
    for clave, valor in registro.items():
        if isinstance(valor, str) and valor.count('/') == 7:
            campos['fen'] = clave
        elif isinstance(valor, list) and valor and isinstance(valor[0], dict):
            campos['evals'] = clave

    faltan = {'fen', 'evals'} - set(campos)
    assert not faltan, (
        f'no encuentro {faltan} en el registro. Esto es lo que trae:\n'
        f'{json.dumps(registro, indent=2)[:800]}'
    )

    evaluacion = registro[campos['evals']][0]
    campos['depth'] = next((k for k, v in evaluacion.items()
                            if k.lower().startswith('depth') and isinstance(v, int)), None)
    campos['pvs'] = next((k for k, v in evaluacion.items()
                          if isinstance(v, list) and v and isinstance(v[0], dict)), None)
    assert campos['pvs'], (
        f'no encuentro las lineas dentro de una evaluacion:\n'
        f'{json.dumps(evaluacion, indent=2)[:600]}'
    )
    linea = evaluacion[campos['pvs']][0]
    assert 'cp' in linea or 'mate' in linea, (
        f'la linea no trae ni cp ni mate:\n{json.dumps(linea, indent=2)}'
    )
    return campos


CAMPOS = detectar(PRIMERO)
for nombre, clave in CAMPOS.items():
    print(f'  {nombre:<8}-> {clave!r}')


def leer(registro: dict):
    """(fen, cp, mate, profundidad, n_evals, n_lineas) de la evaluacion mas profunda."""
    evaluaciones = registro[CAMPOS['evals']]
    clave_depth = CAMPOS['depth']
    mejor = (max(evaluaciones, key=lambda e: e.get(clave_depth, 0))
             if clave_depth else evaluaciones[0])
    lineas_pv = mejor[CAMPOS['pvs']]
    primera_linea = lineas_pv[0]
    return (
        registro[CAMPOS['fen']],
        primera_linea.get('cp'),
        primera_linea.get('mate'),
        mejor.get(clave_depth) if clave_depth else None,
        len(evaluaciones),
        len(lineas_pv),
    )


print()
print('la primera posicion, ya leida:', leer(PRIMERO))

  fen     -> 'fen'
  evals   -> 'evals'
  depth   -> 'depth'
  pvs     -> 'pvs'

la primera posicion, ya leida: ('7r/1p3k2/p1bPR3/5p2/2B2P1p/8/PP4P1/3K4 b - -', 69, None, 46, 1, 5)


## 5. El recorrido

Una sola pasada contando todo lo que hace falta. Lo que se mide acá es lo que
decide la escalera de la notebook 2.

In [7]:
from tqdm.auto import tqdm

from chessdl.data.schema import position_key

profundidades = Counter()
evals_por_posicion = Counter()
lineas_por_eval = Counter()
campos_fen = Counter()
claves = set()

total = mates = sin_puntaje = ilegibles = blancas = intentos_clave = 0
cps = []
sin_comprimir = 0

flujo, contador = abrir(URL)
arranque = time.perf_counter()

with flujo:
    for linea in tqdm(flujo, total=LIMITE, desc='posiciones', unit='pos'):
        if total >= LIMITE:
            break
        sin_comprimir += len(linea)
        try:
            fen, cp, mate, depth, n_evals, n_lineas = leer(json.loads(linea))
        except Exception:
            ilegibles += 1
            continue

        total += 1
        profundidades[depth] += 1
        evals_por_posicion[n_evals] += 1
        lineas_por_eval[n_lineas] += 1
        partes = fen.split(' ')
        campos_fen[len(partes)] += 1
        blancas += partes[1] == 'w'
        if mate is not None:
            mates += 1
        elif cp is None:
            sin_puntaje += 1
        else:
            if len(cps) < 400_000:          # muestra para los percentiles
                cps.append(cp)
        if intentos_clave < 400_000:        # muestra para buscar repetidas
            claves.add(position_key(fen))
            intentos_clave += 1

SEGUNDOS = time.perf_counter() - arranque
COMPRIMIDO_LEIDO = contador.bytes

print(f'\nposiciones leidas   {total:,}')
print(f'ilegibles           {ilegibles:,}')
print(f'mates               {mates:,}  ({mates/total:.2%})')
print(f'sin cp ni mate      {sin_puntaje:,}')
print(f'blancas al turno    {blancas/total:.2%}')
print(f'campos del FEN      {dict(campos_fen)}')
print(f'evals por posicion  {dict(sorted(evals_por_posicion.items())[:6])}')
print(f'lineas por eval     {dict(sorted(lineas_por_eval.items())[:6])}')
print()
print(f'tiempo              {SEGUNDOS/60:.1f} min')
print(f'comprimido leido    {COMPRIMIDO_LEIDO/1e9:.2f} GB'
      f'   ({COMPRIMIDO_LEIDO/SEGUNDOS/1e6:.1f} MB/s)')
print(f'sin comprimir       {sin_comprimir/1e9:.2f} GB'
      f'   (ratio {sin_comprimir/COMPRIMIDO_LEIDO:.1f}x)')
print(f'bytes comprimidos por posicion: {COMPRIMIDO_LEIDO/total:.0f}')

posiciones:   0%|          | 0/2000000 [00:00<?, ?pos/s]


posiciones leidas   2,000,000
ilegibles           0
mates               324,286  (16.21%)
sin cp ni mate      0
blancas al turno    51.74%
campos del FEN      {4: 2000000}
evals por posicion  {1: 1273403, 2: 358432, 3: 252998, 4: 99076, 5: 15433, 6: 633}
lineas por eval     {1: 1145979, 2: 179625, 3: 334527, 4: 56619, 5: 281414, 6: 47}

tiempo              8.8 min
comprimido leido    0.16 GB   (0.3 MB/s)
sin comprimir       0.90 GB   (ratio 5.5x)
bytes comprimidos por posicion: 82


In [8]:
import numpy as np

cps = np.array(cps)
print('PROFUNDIDAD de la evaluacion mas profunda de cada posicion')
print('-' * 58)
acumulado = 0
for d, n in sorted(profundidades.items(), key=lambda kv: -(kv[0] or 0)):
    acumulado += n
    print(f'  depth {str(d):>4}   {n:>10,}  ({n/total:>6.2%})   '
          f'acumulado desde arriba: {acumulado/total:>6.2%}')
print()

print('CUANTAS SOBREVIVEN A CADA FILTRO')
print('-' * 58)
SUPERVIVENCIA = {}
for umbral in (0, 12, 16, 20, 25, 30):
    vivas = sum(n for d, n in profundidades.items() if (d or 0) >= umbral)
    SUPERVIVENCIA[umbral] = vivas / total
    marca = '   <- la del dataset del proyecto' if umbral == 12 else ''
    print(f'  depth >= {umbral:>2}   {vivas/total:>7.2%}{marca}')
print()

print('DISTRIBUCION DE LA EVALUACION (centipeones, sin contar mates)')
print('-' * 58)
for p in (1, 5, 25, 50, 75, 95, 99):
    print(f'  percentil {p:>2}   {np.percentile(cps, p):>8.0f} cp')
print(f'  |cp| > 2000   {np.mean(np.abs(cps) > 2000):.2%}  (se recorta al normalizar)')
print()
repetidas = 1 - len(claves) / intentos_clave
print(f'de {intentos_clave:,} posiciones muestreadas hay {len(claves):,} distintas '
      f'-> {repetidas:.2%} repetidas'
      f"{'' if repetidas < 0.001 else '   <- OJO, el dataset del proyecto deduplica'}")

PROFUNDIDAD de la evaluacion mas profunda de cada posicion
----------------------------------------------------------
  depth  245       76,458  ( 3.82%)   acumulado desde arriba:  3.82%
  depth  240           87  ( 0.00%)   acumulado desde arriba:  3.83%
  depth  235          100  ( 0.01%)   acumulado desde arriba:  3.83%
  depth  230          117  ( 0.01%)   acumulado desde arriba:  3.84%
  depth  225          141  ( 0.01%)   acumulado desde arriba:  3.85%
  depth  220          145  ( 0.01%)   acumulado desde arriba:  3.85%
  depth  215          171  ( 0.01%)   acumulado desde arriba:  3.86%
  depth  210          178  ( 0.01%)   acumulado desde arriba:  3.87%
  depth  205          230  ( 0.01%)   acumulado desde arriba:  3.88%
  depth  200          196  ( 0.01%)   acumulado desde arriba:  3.89%
  depth  195          252  ( 0.01%)   acumulado desde arriba:  3.90%
  depth  190          266  ( 0.01%)   acumulado desde arriba:  3.92%
  depth  185          295  ( 0.01%)   acumulado desde 

### Lo que dice el recorrido

- **Profundidad: el filtro casi no cuesta.** El 99,35 % de las posiciones está
  evaluada a profundidad 12 o más —la del dataset del proyecto— y la mediana
  ronda 24. Exigir 12 descarta menos del 1 %; exigir 20 ya descarta el 12 %.
- **Hay picos que no son profundidades de búsqueda comunes:** 245 (3,8 %), 99
  (2,8 %) y los múltiplos de 5 por encima de 100. Lo más probable es que sean
  posiciones donde el motor llega a su profundidad máxima sin esfuerzo —mates
  forzados, finales muy simples—. Quedan todas por encima de cualquier umbral
  razonable, así que no cambian nada de lo que sigue.
- **Los mates son el 16,2 %**, contra el 1,4 % del dataset del proyecto. Es la
  diferencia más grande entre las dos fuentes, y la que hay que tener presente al
  leer cualquier RMSE: un mate es una etiqueta de ±1, y cuantos más haya, más
  alto queda el piso de la media constante.
- **Sin repetidas** en 400.000 posiciones, y **51,7 % con blancas al turno**: el
  dump no necesita deduplicarse, y el balance de color queda cerca del 50 %
  como pide el requerimiento 1.3 para el dataset del proyecto.


## 6. El dump entero

De lo leído al total, por regla de tres sobre los **bytes comprimidos**: es la
única magnitud que se puede comparar contra el tamaño del archivo.

> **Los tiempos de esta sección son pesimistas.** Los 0,3 MB/s salen de la
> pasada de la sección 5, que descomprime y parsea cada línea en Python mientras
> baja: miden ese bucle, no la red. La notebook 2 baja por rangos de bytes en
> paralelo y parsea después, así que la descarga real es varias veces más
> rápida.

In [9]:
from chessdl.training.cache import cache_size_bytes

assert COMPRIMIDO_TOTAL, 'el servidor no informo el tamaño; no se puede extrapolar'
FACTOR = COMPRIMIDO_TOTAL / COMPRIMIDO_LEIDO
POSICIONES_TOTAL = int(total * FACTOR)
MB_POR_S = COMPRIMIDO_LEIDO / SEGUNDOS / 1e6

print(f'leido            {COMPRIMIDO_LEIDO/1e9:>7.2f} GB  ->  {total:>12,} posiciones')
print(f'archivo entero   {COMPRIMIDO_TOTAL/1e9:>7.2f} GB  ->  {POSICIONES_TOTAL:>12,} posiciones (estimado)')
print(f'descargarlo todo tardaria {COMPRIMIDO_TOTAL/MB_POR_S/1e6/3600:.1f} h a {MB_POR_S:.1f} MB/s')
print()

print('PARA JUNTAR N POSICIONES HAY QUE BAJAR...')
print('-' * 70)
GB_POR_POSICION = COMPRIMIDO_LEIDO / total / 1e9
superv_20 = SUPERVIVENCIA.get(20, 0.0)

print(f"{'objetivo':>9}{'todas: GB':>12}{'min':>7}{'depth>=20: GB':>16}{'min':>7}")
print('-' * 70)
for objetivo in (5_000_000, 10_000_000, 20_000_000, 30_000_000):
    gb = objetivo * GB_POR_POSICION
    fila = f'{objetivo/1e6:>8.0f}M{gb:>12.1f}{gb*1e3/MB_POR_S/60:>7.0f}'
    if superv_20 > 0:
        gb_f = objetivo / superv_20 * GB_POR_POSICION
        cabe = gb_f <= COMPRIMIDO_TOTAL / 1e9
        fila += f'{gb_f:>16.1f}{gb_f*1e3/MB_POR_S/60:>7.0f}'
        fila += '' if cabe else '   NO ALCANZAN'
    else:
        fila += f"{'ninguna llega a 20':>23}"
    print(fila)
print()
print('La columna de la izquierda usa todas las posiciones; la de la derecha,')
print('solo las evaluadas a profundidad 20 o mas. La diferencia entre las dos')
print('es el precio de mejorar las etiquetas.')

leido               0.16 GB  ->     2,000,000 posiciones
archivo entero     22.09 GB  ->   270,043,065 posiciones (estimado)
descargarlo todo tardaria 19.9 h a 0.3 MB/s

PARA JUNTAR N POSICIONES HAY QUE BAJAR...
----------------------------------------------------------------------
 objetivo   todas: GB    min   depth>=20: GB    min
----------------------------------------------------------------------
       5M         0.4     22             0.5     25
      10M         0.8     44             0.9     50
      20M         1.6     88             1.9    101
      30M         2.5    132             2.8    151

La columna de la izquierda usa todas las posiciones; la de la derecha,
solo las evaluadas a profundidad 20 o mas. La diferencia entre las dos
es el precio de mejorar las etiquetas.


## 7. Qué escalera entra

El límite no es el disco: **el caché de tensores tiene que entrar en RAM.** Si
no entra, el entrenamiento lee filas sueltas del disco y la época pasa de 4
minutos a 40 — medido, 1,0 s por lote contra 0,1 s.

A 1.152 bytes por posición, y dejando ~8 GB para Python, el DataFrame y el
modelo.

In [10]:
RAM_GB = 51        # alta RAM en Colab Pro; poner 12.7 para el runtime normal
MARGEN_GB = 8      # python, el dataframe, el modelo, el sistema

print(f'runtime de {RAM_GB} GB, dejando {MARGEN_GB} GB de margen\n')
print(f"{'brazo':>10}{'caché':>10}{'¿entra?':>10}{'¿hay datos (depth>=20)?':>26}")
print('-' * 56)
disponibles = POSICIONES_TOTAL * SUPERVIVENCIA.get(20, 1.0)
for n in (5_000_000, 10_000_000, 20_000_000, 30_000_000, 40_000_000):
    gb = cache_size_bytes(n) / 1e9
    entra = gb <= RAM_GB - MARGEN_GB
    hay = n <= disponibles
    print(f'{n/1e6:>9.0f}M{gb:>9.1f}G{"si" if entra else "NO":>10}'
          f'{("si" if hay else "NO") :>26}')

posibles = [n for n in (5_000_000, 10_000_000, 20_000_000, 30_000_000, 40_000_000)
            if cache_size_bytes(n) / 1e9 <= RAM_GB - MARGEN_GB and n <= disponibles]
print()
if len(posibles) >= 3:
    print(f'Escalera sugerida: {[f"{n/1e6:.0f}M" for n in posibles[-3:]]}')
    horas = sum(n * 0.95 / 1e6 * (3.8 / 2.3) * 10 for n in posibles[-3:]) / 60
    print(f'Costo con EPOCAS = 10: ~{horas:.1f} h de GPU, mas la descarga y el caché.')
else:
    print('No entran tres brazos. Opciones: runtime con mas RAM, o bajar el brazo grande.')

runtime de 51 GB, dejando 8 GB de margen

     brazo     caché   ¿entra?   ¿hay datos (depth>=20)?
--------------------------------------------------------
        5M      5.8G        si                        si
       10M     11.5G        si                        si
       20M     23.0G        si                        si
       30M     34.6G        si                        si
       40M     46.1G        NO                        si

Escalera sugerida: ['10M', '20M', '30M']
Costo con EPOCAS = 10: ~15.7 h de GPU, mas la descarga y el caché.


## 8. Qué sigue

Con estos números se fija la escalera de la **notebook 2 (entrenamiento)**:
brazos encajados, mismos hiperparámetros que `campana2-warmup`, caché en orden
barajado y test reservado.

La escalera que se corrió no es la sugerida por la sección 7 (10, 20 y 30 M),
sino **5, 10 y 30 M con la red de la memoria, más un cuarto brazo de 30 M con
una red el doble de profunda**. Dos motivos: los volúmenes quedan espaciados en
escala logarítmica —que es como se lee una curva de aprendizaje—, y el cuarto
brazo responde una pregunta que la escalera sola no puede, si con todos los
datos la red se queda corta de capacidad.

Dos cosas que este survey deja decididas y que conviene anotar antes de seguir:

- **El filtro de profundidad.** Se exige **12**, la misma profundidad con la
  que se etiquetó el dataset del proyecto: cuesta menos del 1 % del volumen y
  garantiza que ninguna etiqueta del apéndice sea más superficial que las de la
  memoria.
- **De qué lado viene el `cp`.** El survey no lo resuelve: lo resuelve el
  detector de signo de la notebook 2, correlacionando la etiqueta con el
  material de las blancas por separado según quién mueve. Se menciona acá
  porque es el error que no falla — entrena igual y devuelve un modelo que
  evalúa al revés la mitad de las posiciones. (Resultado: el `cp` y el `mate`
  vienen en perspectiva de las **blancas**.)

> **Una salvedad sobre el muestreo.** Este survey lee desde el **principio** del
> archivo, porque un stream zstd no se puede saltear. Si el dump tiene algún
> orden interno, las primeras posiciones pueden no representar al resto. La
> notebook 2 lee un prefijo por el mismo motivo, así que al menos el sesgo es el
> mismo en las dos; pero es una limitación real y va dicha en el apéndice.

**Próximo paso:** [`lichess_2_entrenamiento.ipynb`](lichess_2_entrenamiento.ipynb),
que entrena los cuatro brazos sobre este dump.
